# CNN Exercise

In [ ]:
import ipywidgets as widgets
import io
import requests
import torch
import matplotlib.pyplot as plt
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from IPython.display import display
from PIL import Image

### Create MobileNetV3 model

In [ ]:
weights = MobileNet_V3_Small_Weights.DEFAULT
model = mobilenet_v3_small(weights = weights)
model.eval()

labels = requests.get("https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt").text.splitlines()

preprocess = weights.transforms()

### Load image

In [ ]:
upload = widgets.FileUpload(
	accept="image/*",
	multiple=False
)

display(upload)

In [ ]:
# cause colab and jupyter use different ipywidgets version
def getFirstProp(value):
	if isinstance(value, dict):
		propName = list(value.keys())[0]
		return value[propName]
	else:
		return value[0]

if len(upload.value)>0: 
	content = getFirstProp(upload.value)["content"]
	image = Image.open(io.BytesIO(content))
	display(image)
else:
	print("Upload a file first!")

### Apply model to image

In [ ]:
batch = preprocess(image).unsqueeze(0)
with torch.no_grad():
	prediction = model(batch).squeeze(0).softmax(0)
    
# get top 10 labels
top10Prob, top10Indices = torch.topk(prediction, 10)
for i in top10Indices:
	print(labels[i])

### Display image and labels

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Show the actual image
ax1.imshow(image)
ax1.axis('off')
ax1.set_title("Input Image")

# Show the Bar Chart
topLabels = [labels[idx] for idx in top10Indices]
topProbs = [prob.item() for prob in top10Prob]

ax2.barh(topLabels, topProbs, color='skyblue')
ax2.invert_yaxis()  # Put the highest probability at the top
ax2.set_xlabel('Probabilities')
ax2.set_title('Top 10 Predictions')

plt.tight_layout()
plt.show()